In [3]:
import numpy as np
from adaline import AdalineGD

In [4]:
rng = np.random.RandomState(42)
n = 400

In [5]:
glucose = rng.normal(120, 30, n).clip(70, 200)
bmi = rng.normal(30, 6, n).clip(18, 45)
age = rng.normal(45, 14, n).clip(21, 80)
blood_pressure = rng.normal(72, 12, n).clip(50, 110)

In [6]:
risk = (0.05 * (glucose - 120)
        + 0.10 * (bmi - 30)
        + 0.04 * (age - 45)
        + rng.normal(0, 1.0, n))
y = (risk > 0).astype(int)

In [7]:
X = np.column_stack([glucose, bmi, age, blood_pressure])

In [9]:
print(f"\n  diabetic (1):     {int(y.sum())} patients")
print(f"  non-diabetic (0): {int((y == 0).sum())} patients")
print(f"\nFeature ranges:")
for name, col in zip(["glucose", "bmi", "age", "blood_pressure"], X.T):
    print(f"  {name:16s} {col.min():6.1f} to {col.max():6.1f}")



  diabetic (1):     217 patients
  non-diabetic (0): 183 patients

Feature ranges:
  glucose            70.0 to  200.0
  bmi                18.0 to   45.0
  age                21.0 to   80.0
  blood_pressure     50.0 to  103.2


In [10]:
ada_raw = AdalineGD(eta=0.00001, n_iter=50).fit(X, y)
print(f"  epoch 1 loss:  {ada_raw.losses_[0]:.4f}")
print(f"  epoch 50 loss: {ada_raw.losses_[-1]:.4f}")
print(f"  accuracy: {(ada_raw.predict(X) == y).mean():.1%}")

  epoch 1 loss:  0.2676
  epoch 50 loss: 0.1921
  accuracy: 73.8%


In [12]:
means = X.mean(axis=0)
stds =  X.std(axis=0)
X_std = (X - means) / stds

In [13]:
for ne in [50, 100, 150, 300]:
    a = AdalineGD(eta=0.01, n_iter=ne).fit(X_std, y)
    acc = (a.predict(X_std) == y).mean()
    print(f"  {ne:3d} epochs -> loss {a.losses_[-1]:.3f}, accuracy {acc:.1%}")

   50 epochs -> loss 0.191, accuracy 66.2%
  100 epochs -> loss 0.141, accuracy 81.5%
  150 epochs -> loss 0.134, accuracy 83.5%
  300 epochs -> loss 0.133, accuracy 83.0%


In [14]:
ada = AdalineGD(eta=0.01, n_iter=150).fit(X_std, y)

In [15]:
new_patient = np.array([[185, 38, 60, 85]])

In [17]:
new_std = (new_patient - means) / stds
pred = ada.predict(new_std)[0]

In [18]:
healthy = np.array([[90, 22, 28, 68]])
healthy_std = (healthy - means) / stds
pred2 = ada.predict(healthy_std)[0]